# 01. Carga y validación del dataset Elliptic

Este notebook tiene como objetivo cargar los tres archivos principales del Elliptic Data Set, validar su estructura inicial y generar archivos intermedios en formato Parquet para los siguientes niveles metodológicos del proyecto.

In [2]:
# Imports y rutas
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from aml_gnn.utils.paths import RAW_DIR, INTERIM_DIR
from aml_gnn.data.load_elliptic import load_raw_elliptic
from aml_gnn.data.validate_schema import validate_elliptic_shapes
from aml_gnn.data.preprocess import (
    preprocess_elliptic_features,
    preprocess_elliptic_classes,
    merge_features_classes
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)

PROJECT_ROOT: /home/lucho/aml-gnn-tesis
RAW_DIR: /home/lucho/aml-gnn-tesis/data/raw/elliptic
INTERIM_DIR: /home/lucho/aml-gnn-tesis/data/interim/elliptic


In [3]:
# verificando los archivos esperados
expected_files = [
    "elliptic_txs_features.csv",
    "elliptic_txs_classes.csv",
    "elliptic_txs_edgelist.csv"
]

for file_name in expected_files:
    file_path = RAW_DIR / file_name
    print(file_name, "->", file_path.exists())

elliptic_txs_features.csv -> True
elliptic_txs_classes.csv -> True
elliptic_txs_edgelist.csv -> True


In [4]:
# Carga de los datos
features_raw, classes_raw, edges_raw = load_raw_elliptic(RAW_DIR)

print("Features:", features_raw.shape)
print("Classes:", classes_raw.shape)
print("Edges:", edges_raw.shape)

Features: (203769, 167)
Classes: (203769, 2)
Edges: (234355, 2)


In [5]:
# Validando la estructura inicial
schema_report = validate_elliptic_shapes(
    features_raw,
    classes_raw,
    edges_raw
)

schema_report

{'features_shape': (203769, 167),
 'classes_shape': (203769, 2),
 'edges_shape': (234355, 2),
 'features_columns': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  

VISTA PREVIA DE LOS ARCHIVOS (de las primeras filas)

In [6]:
features_raw.head()

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [7]:
classes_raw.head()

,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


In [8]:
edges_raw.head()

,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


In [ ]:
# Preprocesando los features y clases
features_df = preprocess_elliptic_features(features_raw)
classes_df = preprocess_elliptic_classes(classes_raw)

print("Features procesadas:", features_df.shape)
print("Clases procesadas:", classes_df.shape)

Features procesadas: (203769, 167)
Clases procesadas: (203769, 3)


In [10]:
# Revisando etiquetas
classes_df["label"].value_counts(dropna=False)

label
unknown    157205
licit       42019
illicit      4545
Name: count, dtype: int64

In [11]:
# Uniendo los features con las etiquetas
nodes_df = merge_features_classes(features_df, classes_df)

print("Nodes:", nodes_df.shape)
nodes_df.head()

Nodes: (203769, 168)


,txId,timestep,f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,...,f_156,f_157,f_158,f_159,f_160,f_161,f_162,f_163,f_164,label
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,unknown
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,unknown
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792,unknown
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792,licit
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117,unknown


In [12]:
# Realizando algunas validaciones
print("Nodos únicos en features:", features_df["txId"].nunique())
print("Nodos únicos en classes:", classes_df["txId"].nunique())
print("Nodos únicos en nodes_df:", nodes_df["txId"].nunique())

print("Duplicados en txId:", nodes_df["txId"].duplicated().sum())
print("Valores nulos en label:", nodes_df["label"].isna().sum())

Nodos únicos en features: 203769
Nodos únicos en classes: 203769
Nodos únicos en nodes_df: 203769
Duplicados en txId: 0
Valores nulos en label: 0


In [13]:
class_summary = (
    nodes_df["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

class_summary["percentage"] = (
    class_summary["count"] / class_summary["count"].sum() * 100
).round(2)

class_summary

,label,count,percentage
0,unknown,157205,77.15
1,licit,42019,20.62
2,illicit,4545,2.23


In [14]:
# Guardando archivos
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

nodes_path = INTERIM_DIR / "nodes_interim.parquet"
edges_path = INTERIM_DIR / "edges_interim.parquet"

nodes_df.to_parquet(nodes_path, index=False)
edges_raw.to_parquet(edges_path, index=False)

print("Guardado:", nodes_path)
print("Guardado:", edges_path)

Guardado: /home/lucho/aml-gnn-tesis/data/interim/elliptic/nodes_interim.parquet
Guardado: /home/lucho/aml-gnn-tesis/data/interim/elliptic/edges_interim.parquet
